In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical


In [ ]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# OPTION 2: CIFAR-10 .npz file in the same folder (OFFLINE)
# Comment the line above and uncomment all lines below.
# import numpy as np
# data = np.load("cifar10.npz")
# x_train, y_train = data["x_train"], data["y_train"]
# x_test, y_test = data["x_test"], data["y_test"]

# OPTION 3: CIFAR-10 .csv files in the same folder (OFFLINE)
# Each CSV: first column = label, next 3072 columns = pixel values.
# Comment the line above and uncomment all lines below.
# import pandas as pd
# train = pd.read_csv("cifar10_train.csv")  # add header=None if no header row
# test = pd.read_csv("cifar10_test.csv")    # add header=None if no header row
# y_train = train.iloc[:, 0].to_numpy()
# x_train = train.iloc[:, 1:].to_numpy().reshape(-1, 32, 32, 3)
# y_test = test.iloc[:, 0].to_numpy()
# x_test = test.iloc[:, 1:].to_numpy().reshape(-1, 32, 32, 3)
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [ ]:
base_model = VGG16(weights='imagenet', include_top=False,
                   input_shape=(32, 32, 3))
for layer in base_model.layers:
    layer.trainable = False


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model = models.Sequential()
model.add(base_model)
model.add(layers.Flatten())
model.add(layers.Dense(256, activation='relu'))
model.add(layers.Dropout(0.5))
model.add(layers.Dense(10, activation='softmax'))


In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
print(model.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 1, 1, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,848,586 (56.64 MB)

 Trainable params: 133,898 (523.04 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

None


In [ ]:
history = model.fit(x_train, y_train,
                    epochs=5,
                    batch_size=64,
                    validation_data=(x_test, y_test))


Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 852s 1s/step - accuracy: 0.3929 - loss: 1.7157 - val_accuracy: 0.5497 - val_loss: 1.2894
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 841s 1s/step - accuracy: 0.5401 - loss: 1.3168 - val_accuracy: 0.5679 - val_loss: 1.2274
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 871s 1s/step - accuracy: 0.5582 - loss: 1.2668 - val_accuracy: 0.5813 - val_loss: 1.1979
Epoch 4/5
531/782 ━━━━━━━━━━━━━━━━━━━━ 3:40 880ms/step - accuracy: 0.5758 - loss: 1.2152

In [ ]:
for layer in base_model.layers[-4:]:  # last 4 conv layers
    layer.trainable = True


In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),  # lower LR
              loss='categorical_crossentropy',
              metrics=['accuracy'])


In [ ]:
history_fine = model.fit(x_train, y_train,
                         epochs=3,
                         batch_size=64,
                         validation_data=(x_test, y_test))


In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test accuracy after fine-tuning:", test_acc)
